<a href="https://colab.research.google.com/github/BradyStonk/BradyStonk-Growth-Investing-System/blob/main/%E3%80%8CAlpha_Attack_Research_UI_v2_1_Stable_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha Attack Research UI v2

整合：

- Model 4 Attack Signal
- Clock D
- Model 5 Return Percentile
- Profit Density
- 最終風險狀態

操作流程：

1. 輸入 Ticker、Benchmark、日期。
2. 按「執行 Model 4」。
3. 從下拉選單選擇本輪初始 Attack Signal。
4. 按「計算 Clock D + Model 5」。


In [ ]:
# 安裝必要套件（Colab 第一次執行即可）
!pip -q install yfinance ipywidgets pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.8 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from datetime import date, timedelta
import math

import numpy as np
import pandas as pd
import yfinance as yf
import ipywidgets as widgets

from IPython.display import display, clear_output


# =============================================================================
# Model 5 frozen thresholds
# =============================================================================

RETURN_PERCENTILE_TRIGGER = 80.0
DENSITY_NEAR_TRIGGER = 1.50
DENSITY_TRIGGER = 1.60


# =============================================================================
# Data structures
# =============================================================================

@dataclass(frozen=True)
class ClockResult:
    initial_signal_date: pd.Timestamp
    first_failure_date: pd.Timestamp | None
    clock_start_date: pd.Timestamp
    clock_start_price: float
    confirmation_signal_date: pd.Timestamp | None
    trading_days_at_confirmation: int | None


class Model5Status(str, Enum):
    CONTINUE = "CONTINUE"
    MATURE = "MATURE"
    ALERT = "ALERT"
    HIGH_RISK_WATCH = "HIGH_RISK_WATCH"
    HIGH_RISK = "HIGH_RISK"


@dataclass(frozen=True)
class Model5Result:
    ticker: str
    attack_start_date: str
    attack_start_price: float
    current_price: float
    attack_trading_days: int
    attack_return_pct: float
    attack_return_percentile: float
    profit_density_pct_per_day: float
    return_triggered: bool
    density_near_triggered: bool
    density_triggered: bool
    status: Model5Status
    action: str
    explanation: str


# =============================================================================
# Data download
# =============================================================================

def download_daily(
    ticker: str,
    start: str,
    end: str,
) -> pd.DataFrame:
    ticker = ticker.upper().strip()

    if not ticker:
        raise ValueError("Ticker 不可為空。")

    data = yf.download(
        ticker,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        progress=False,
        actions=False,
    )

    if data.empty:
        raise RuntimeError(f"無法取得 {ticker} 資料。")

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    if "Adj Close" not in data.columns:
        data["Adj Close"] = data["Close"]

    required = ["High", "Low", "Close", "Adj Close"]
    missing = [
        column
        for column in required
        if column not in data.columns
    ]

    if missing:
        raise RuntimeError(
            f"{ticker} 缺少必要欄位：{missing}"
        )

    return data.sort_index()


# =============================================================================
# Model 4
# =============================================================================

def prepare_model4(
    stock: pd.DataFrame,
    benchmark: pd.DataFrame,
    range_window: int = 5,
    prior_range_window: int = 20,
    rs_window: int = 10,
) -> pd.DataFrame:

    stock_df = stock[
        ["High", "Low", "Close", "Adj Close"]
    ].copy()

    stock_df.columns = [
        "Stock High",
        "Stock Low",
        "Stock Close",
        "Stock Adj Close",
    ]

    benchmark_df = benchmark[
        ["Adj Close"]
    ].copy()

    benchmark_df.columns = [
        "Benchmark Adj Close"
    ]

    df = (
        stock_df
        .join(
            benchmark_df,
            how="inner",
        )
        .dropna()
        .copy()
    )

    df["Daily Range %"] = (
        (
            df["Stock High"]
            - df["Stock Low"]
        )
        / df["Stock Close"]
        * 100.0
    )

    df["5D Avg Range %"] = (
        df["Daily Range %"]
        .rolling(range_window)
        .mean()
    )

    df["Prior 20D Avg Range %"] = (
        df["Daily Range %"]
        .shift(range_window)
        .rolling(prior_range_window)
        .mean()
    )

    df["Volatility Contraction"] = (
        df["5D Avg Range %"]
        < df["Prior 20D Avg Range %"]
    )

    df["Stock 10D Return %"] = (
        df["Stock Adj Close"]
        .pct_change(rs_window)
        * 100.0
    )

    df["Benchmark 10D Return %"] = (
        df["Benchmark Adj Close"]
        .pct_change(rs_window)
        * 100.0
    )

    df["10D Relative Strength %"] = (
        df["Stock 10D Return %"]
        - df["Benchmark 10D Return %"]
    )

    df["RS Strong"] = (
        df["10D Relative Strength %"] > 0
    )

    df["Attack Condition"] = (
        df["Volatility Contraction"]
        & df["RS Strong"]
    )

    previous_condition = (
        df["Attack Condition"]
        .shift(1)
        .fillna(False)
        .astype(bool)
    )

    df["Attack Signal"] = (
        df["Attack Condition"]
        & ~previous_condition
    )

    return df


def build_signal_table(
    df: pd.DataFrame,
) -> pd.DataFrame:

    signal_df = df.loc[
        df["Attack Signal"].fillna(False),
        [
            "Stock Adj Close",
            "5D Avg Range %",
            "Prior 20D Avg Range %",
            "Stock 10D Return %",
            "Benchmark 10D Return %",
            "10D Relative Strength %",
        ],
    ].copy()

    signal_df.insert(
        0,
        "Signal Date",
        signal_df.index.strftime("%Y-%m-%d"),
    )

    signal_df.insert(
        2,
        "Signal Number",
        range(1, len(signal_df) + 1),
    )

    return signal_df.reset_index(drop=True)


# =============================================================================
# Clock D
# =============================================================================

def calculate_clock_d(
    df: pd.DataFrame,
    initial_signal_date: str | pd.Timestamp,
) -> ClockResult:

    initial = pd.Timestamp(
        initial_signal_date
    )

    if initial not in df.index:
        raise ValueError(
            f"資料中找不到初始訊號日期："
            f"{initial.date()}"
        )

    if not bool(
        df.loc[
            initial,
            "Attack Signal",
        ]
    ):
        raise ValueError(
            f"{initial.date()} 不是 Attack Signal。"
        )

    after_initial = df.loc[
        df.index > initial
    ].copy()

    failures = after_initial.loc[
        ~after_initial[
            "Attack Condition"
        ].fillna(False)
    ]

    first_failure = (
        failures.index[0]
        if not failures.empty
        else None
    )

    confirmation = None

    if first_failure is not None:

        confirmations = df.loc[
            (
                df.index > first_failure
            )
            &
            (
                df["Attack Signal"]
                .fillna(False)
            )
        ]

        if not confirmations.empty:
            confirmation = (
                confirmations.index[0]
            )

    clock_start = initial
    clock_price = float(
        df.loc[
            initial,
            "Stock Adj Close",
        ]
    )

    if (
        first_failure is not None
        and confirmation is not None
    ):

        trough_window = df.loc[
            (
                df.index >= first_failure
            )
            &
            (
                df.index < confirmation
            )
        ].copy()

        if not trough_window.empty:

            clock_start = (
                trough_window[
                    "Stock Adj Close"
                ]
                .idxmin()
            )

            clock_price = float(
                trough_window.loc[
                    clock_start,
                    "Stock Adj Close",
                ]
            )

    elapsed = None

    if confirmation is not None:
        elapsed = (
            df.index.get_loc(
                confirmation
            )
            -
            df.index.get_loc(
                clock_start
            )
        )

    return ClockResult(
        initial_signal_date=initial,
        first_failure_date=first_failure,
        clock_start_date=clock_start,
        clock_start_price=clock_price,
        confirmation_signal_date=confirmation,
        trading_days_at_confirmation=elapsed,
    )


# =============================================================================
# Model 5
# =============================================================================

def calculate_percentile_rank(
    current_value: float,
    historical_values: list[float],
) -> float:

    clean_values = [
        float(value)
        for value in historical_values
        if math.isfinite(float(value))
    ]

    if not clean_values:
        raise ValueError(
            "缺少有效歷史 Attack Return。"
        )

    below_count = sum(
        value < current_value
        for value in clean_values
    )

    equal_count = sum(
        math.isclose(
            value,
            current_value,
            rel_tol=1e-12,
            abs_tol=1e-12,
        )
        for value in clean_values
    )

    return (
        below_count
        + 0.5 * equal_count
    ) / len(clean_values) * 100.0


def determine_status(
    return_triggered: bool,
    density_near_triggered: bool,
    density_triggered: bool,
) -> tuple[
    Model5Status,
    str,
    str,
]:

    if (
        return_triggered
        and density_triggered
    ):
        return (
            Model5Status.HIGH_RISK,
            "考慮降槓桿或分批獲利。",
            (
                "Attack Return 已進入自身歷史前 20%，"
                "且 Profit Density 已達正式過熱區。"
            ),
        )

    if (
        return_triggered
        and density_near_triggered
    ):
        return (
            Model5Status.HIGH_RISK_WATCH,
            (
                "停止增加槓桿，"
                "密切監控並準備獲利了結。"
            ),
            (
                "Attack Return 已成熟，"
                "Profit Density 已接近正式過熱門檻。"
            ),
        )

    if return_triggered:
        return (
            Model5Status.MATURE,
            "繼續持有，但每日監控 Profit Density。",
            (
                "Attack Return 已進入自身歷史前 20%，"
                "但目前速度尚未接近過熱。"
            ),
        )

    if density_near_triggered:
        return (
            Model5Status.ALERT,
            (
                "啟動警報，但不因 Density "
                "單獨觸發而退出。"
            ),
            (
                "Profit Density 已快速升高，"
                "但 Attack Return 尚未進入成熟區。"
            ),
        )

    return (
        Model5Status.CONTINUE,
        "維持原策略。",
        (
            "Attack Return 與 Profit Density "
            "均未進入警示區。"
        ),
    )


def evaluate_model5(
    ticker: str,
    clock: ClockResult,
    df: pd.DataFrame,
) -> Model5Result:

    current_price = float(
        df["Stock Adj Close"].iloc[-1]
    )

    current_position = len(df) - 1

    start_position = df.index.get_loc(
        clock.clock_start_date
    )

    attack_days = (
        current_position
        - start_position
    )

    attack_return = (
        current_price
        / clock.clock_start_price
        - 1.0
    ) * 100.0

    density = (
        math.nan
        if attack_days == 0
        else attack_return
        / attack_days
    )

    episode_df = df.loc[
        clock.clock_start_date:
    ].copy()

    historical_returns = (
        (
            episode_df[
                "Stock Adj Close"
            ]
            / clock.clock_start_price
            - 1.0
        )
        * 100.0
    ).tolist()

    percentile = (
        calculate_percentile_rank(
            attack_return,
            historical_returns,
        )
    )

    return_triggered = (
        percentile
        >= RETURN_PERCENTILE_TRIGGER
    )

    density_near_triggered = (
        math.isfinite(density)
        and density
        >= DENSITY_NEAR_TRIGGER
    )

    density_triggered = (
        math.isfinite(density)
        and density
        >= DENSITY_TRIGGER
    )

    status, action, explanation = (
        determine_status(
            return_triggered,
            density_near_triggered,
            density_triggered,
        )
    )

    return Model5Result(
        ticker=ticker.upper().strip(),
        attack_start_date=(
            clock.clock_start_date
            .strftime("%Y-%m-%d")
        ),
        attack_start_price=(
            clock.clock_start_price
        ),
        current_price=current_price,
        attack_trading_days=attack_days,
        attack_return_pct=attack_return,
        attack_return_percentile=percentile,
        profit_density_pct_per_day=density,
        return_triggered=return_triggered,
        density_near_triggered=(
            density_near_triggered
        ),
        density_triggered=(
            density_triggered
        ),
        status=status,
        action=action,
        explanation=explanation,
    )


print("Alpha Attack Engine 載入完成。")

Alpha Attack Engine 載入完成。


In [ ]:
# =============================================================================
# Alpha Attack Research UI v2
# 只需要操作這一格
# =============================================================================

ticker_widget = widgets.Text(
    value="SOXX",
    description="Ticker:",
    layout=widgets.Layout(
        width="320px",
    ),
)

benchmark_widget = widgets.Text(
    value="QQQ",
    description="Benchmark:",
    layout=widgets.Layout(
        width="320px",
    ),
)

start_widget = widgets.DatePicker(
    description="開始日期:",
    value=date(2022, 1, 1),
)

end_widget = widgets.DatePicker(
    description="結束日期:",
    value=date.today(),
)

run_model4_button = widgets.Button(
    description="1. 執行 Model 4",
    button_style="primary",
    icon="play",
)

signal_dropdown = widgets.Dropdown(
    options=[],
    description="初始訊號:",
    layout=widgets.Layout(
        width="420px",
    ),
    disabled=True,
)

run_model5_button = widgets.Button(
    description="2. 計算 Clock D + Model 5",
    button_style="success",
    icon="check",
    disabled=True,
)

model4_output = widgets.Output()
model5_output = widgets.Output()

state = {
    "df": None,
    "signals": None,
}


def run_model4_clicked(_):

    with model4_output:

        clear_output()

        try:

            ticker = (
                ticker_widget.value
                .upper()
                .strip()
            )

            benchmark = (
                benchmark_widget.value
                .upper()
                .strip()
            )

            start_date = (
                start_widget.value
            )

            end_date = (
                end_widget.value
            )

            if start_date is None:
                raise ValueError(
                    "請輸入開始日期。"
                )

            if end_date is None:
                raise ValueError(
                    "請輸入結束日期。"
                )

            print(
                f"下載 {ticker} 與 {benchmark}..."
            )

            stock = download_daily(
                ticker,
                str(start_date),
                str(
                    end_date
                    + timedelta(days=1)
                ),
            )

            benchmark_data = download_daily(
                benchmark,
                str(start_date),
                str(
                    end_date
                    + timedelta(days=1)
                ),
            )

            df = prepare_model4(
                stock,
                benchmark_data,
            )

            signals = build_signal_table(
                df
            )

            state["df"] = df
            state["signals"] = signals

            print()
            print(
                f"標的：{ticker}"
            )
            print(
                f"Benchmark：{benchmark}"
            )
            print(
                f"資料期間："
                f"{df.index[0].date()} "
                f"至 {df.index[-1].date()}"
            )
            print(
                f"Attack Signal 次數："
                f"{len(signals)}"
            )

            if signals.empty:

                signal_dropdown.options = []
                signal_dropdown.disabled = True
                run_model5_button.disabled = True

                print()
                print(
                    "目前期間沒有 Attack Signal。"
                )

                return

            display(
                signals.tail(30)
            )

            signal_dates = (
                signals[
                    "Signal Date"
                ]
                .tolist()
            )

            signal_dropdown.options = (
                signal_dates
            )

            signal_dropdown.value = (
                signal_dates[-1]
            )

            signal_dropdown.disabled = False
            run_model5_button.disabled = False

            print()
            print(
                "請從下拉選單選擇本輪 "
                "Attack 的初始訊號，"
                "再按第二個按鈕。"
            )

        except Exception as exc:

            print(
                f"錯誤：{exc}"
            )


def run_model5_clicked(_):

    with model5_output:

        clear_output()

        try:

            df = state["df"]

            if df is None:
                raise RuntimeError(
                    "請先執行 Model 4。"
                )

            signal_date = (
                signal_dropdown.value
            )

            if signal_date is None:
                raise RuntimeError(
                    "請選擇初始訊號。"
                )

            ticker = (
                ticker_widget.value
                .upper()
                .strip()
            )

            clock = calculate_clock_d(
                df,
                signal_date,
            )

            result = evaluate_model5(
                ticker,
                clock,
                df,
            )

            clock_df = pd.DataFrame(
                [
                    {
                        "Initial Signal":
                            clock.initial_signal_date
                            .strftime("%Y-%m-%d"),

                        "First Failure":
                            (
                                clock.first_failure_date
                                .strftime("%Y-%m-%d")
                                if clock.first_failure_date
                                is not None
                                else None
                            ),

                        "Clock Start":
                            clock.clock_start_date
                            .strftime("%Y-%m-%d"),

                        "Clock Start Price":
                            clock.clock_start_price,

                        "Confirmation":
                            (
                                clock.confirmation_signal_date
                                .strftime("%Y-%m-%d")
                                if clock.confirmation_signal_date
                                is not None
                                else None
                            ),

                        "Trading Days at Confirmation":
                            clock.trading_days_at_confirmation,
                    }
                ]
            )

            print("=" * 78)
            print(
                f"{ticker} — Clock D"
            )
            print("=" * 78)

            display(
                clock_df.round(3)
            )

            print()
            print("=" * 78)
            print(
                f"{ticker} — Model 5 Result"
            )
            print("=" * 78)

            result_df = pd.DataFrame(
                [
                    {
                        "Current Price":
                            result.current_price,

                        "Attack Days":
                            result.attack_trading_days,

                        "Attack Return %":
                            result.attack_return_pct,

                        "Return Percentile":
                            result.attack_return_percentile,

                        "Profit Density %/Day":
                            result.profit_density_pct_per_day,

                        "Status":
                            result.status.value,
                    }
                ]
            )

            display(
                result_df.round(3)
            )

            print()
            print(
                f"STATUS：{result.status.value}"
            )
            print(
                f"ACTION：{result.action}"
            )
            print(
                f"EXPLANATION："
                f"{result.explanation}"
            )

        except Exception as exc:

            print(
                f"錯誤：{exc}"
            )


run_model4_button.on_click(
    run_model4_clicked
)

run_model5_button.on_click(
    run_model5_clicked
)


ui = widgets.VBox(
    [
        widgets.HTML(
            """
            <h2>Alpha Attack Research UI v2</h2>
            <p>
            輸入標的與 Benchmark，
            系統會自動執行 Model 4、
            Clock D 與 Model 5。
            </p>
            """
        ),

        widgets.HBox(
            [
                ticker_widget,
                benchmark_widget,
            ]
        ),

        widgets.HBox(
            [
                start_widget,
                end_widget,
            ]
        ),

        run_model4_button,

        model4_output,

        widgets.HTML(
            "<hr>"
        ),

        signal_dropdown,

        run_model5_button,

        model5_output,
    ]
)

display(ui)